# Analyse détaillée des trajectoires

Analyse approfondie des comportements du modèle DQN.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

from src.envs import LargeTaxiEnv
from src.agent import DQNTaxiAgent
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Charger le modèle

In [ ]:
env = LargeTaxiEnv(render_mode=None, grid_width=15, grid_height=10, n_passengers=2)
agent = DQNTaxiAgent(env=env)
agent.load("../training/checkpoints/dqn_taxi_final.zip")
print("Modèle chargé")

## 2. Collecter des trajectoires détaillées

In [ ]:
def collect_trajectories(agent, env, n_episodes=50):
    """Collecte des trajectoires détaillées."""
    trajectories = []
    
    for ep in range(n_episodes):
        obs, _ = env.reset()
        trajectory = {
            'episode': ep,
            'total_reward': 0,
            'steps': 0,
            'success': False,
            'positions': [],
            'actions': [],
            'rewards': [],
        }
        
        done = False
        while not done:
            # Enregistrer position
            taxi_x, taxi_y = int(obs[0]), int(obs[1])
            trajectory['positions'].append((taxi_x, taxi_y))
            
            # Prédire action
            action = agent.predict(obs, deterministic=True)
            trajectory['actions'].append(action)
            
            # Exécuter
            obs, reward, terminated, truncated, _ = env.step(action)
            trajectory['rewards'].append(reward)
            trajectory['total_reward'] += reward
            trajectory['steps'] += 1
            
            done = terminated or truncated
            if terminated:
                trajectory['success'] = True
        
        trajectories.append(trajectory)
    
    return trajectories

print("Collecte des trajectoires...")
trajectories = collect_trajectories(agent, env, n_episodes=50)
print(f"Collecté {len(trajectories)} trajectoires")

## 3. Statistiques par trajectoire

In [ ]:
# Créer un DataFrame
df = pd.DataFrame([
    {
        'Episode': t['episode'],
        'Total Reward': t['total_reward'],
        'Steps': t['steps'],
        'Success': t['success'],
    }
    for t in trajectories
])

print("\n" + "="*60)
print("STATISTIQUES GLOBALES")
print("="*60)
print(f"Taux de succès : {df['Success'].sum() / len(df) * 100:.1f}%")
print(f"Récompense moyenne : {df['Total Reward'].mean():.2f} ± {df['Total Reward'].std():.2f}")
print(f"Pas moyens : {df['Steps'].mean():.1f} ± {df['Steps'].std():.1f}")
print(f"Min/Max récompense : {df['Total Reward'].min():.0f} / {df['Total Reward'].max():.0f}")
print(f"Min/Max pas : {df['Steps'].min()} / {df['Steps'].max()}")
print("="*60)

print("\nPremières trajectoires :")
print(df.head(10))

## 4. Analyse des succès vs échecs

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Succès vs Échecs
success_counts = df['Success'].value_counts()
colors = ['#06A77D', '#D62828']
axes[0].bar(['Succès', 'Échecs'], [success_counts.get(True, 0), success_counts.get(False, 0)], 
            color=colors, alpha=0.8, edgecolor='white', linewidth=2)
axes[0].set_ylabel('Nombre d\'épisodes', fontsize=11, fontweight='bold')
axes[0].set_title('Succès vs Échecs', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

# Récompense par succès/échec
success_rewards = df[df['Success']]['Total Reward']
fail_rewards = df[~df['Success']]['Total Reward']

axes[1].boxplot([success_rewards, fail_rewards], labels=['Succès', 'Échecs'],
                patch_artist=True,
                boxprops=dict(facecolor='#2E86AB', alpha=0.6),
                medianprops=dict(color='#D62828', linewidth=2))
axes[1].set_ylabel('Récompense totale', fontsize=11, fontweight='bold')
axes[1].set_title('Distribution des récompenses', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../logs/analysis_success_vs_fail.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Visualiser une trajectoire réussie

In [ ]:
# Trouver une trajectoire réussie
successful_trajectories = [t for t in trajectories if t['success']]
if successful_trajectories:
    traj = successful_trajectories[0]
    
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Grille
    ax.set_xlim(-0.5, 14.5)
    ax.set_ylim(-0.5, 9.5)
    ax.set_aspect('equal')
    ax.invert_yaxis()
    
    # Zones
    pickup_zones = [(2, 2), (12, 8)]
    dropoff_zones = [(12, 2), (2, 8)]
    
    for x, y in pickup_zones:
        ax.plot(x, y, 'b*', markersize=25, label='Pickup' if x == 2 else '')
    
    for x, y in dropoff_zones:
        ax.plot(x, y, 'g^', markersize=20, label='Dropoff' if x == 12 else '')
    
    # Trajectoire
    positions = np.array(traj['positions'])
    ax.plot(positions[:, 0], positions[:, 1], 'r-', alpha=0.5, linewidth=1.5, label='Chemin')
    ax.plot(positions[0, 0], positions[0, 1], 'go', markersize=12, label='Départ')
    ax.plot(positions[-1, 0], positions[-1, 1], 'rs', markersize=12, label='Fin')
    
    ax.set_xlabel('Colonne', fontsize=11, fontweight='bold')
    ax.set_ylabel('Ligne', fontsize=11, fontweight='bold')
    ax.set_title(f'Trajectoire réussie — Episode {traj["episode"]} (Reward: {traj["total_reward"]:.0f}, Steps: {traj["steps"]})',
                fontsize=13, fontweight='bold')
    ax.legend(fontsize=10, loc='upper right')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../logs/analysis_successful_trajectory.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("Aucune trajectoire réussie trouvée")

In [ ]:
env.close()
print("Analyse terminée !")